# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get dataset metadata (as one object)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

- Each **Record Set** corresponds to a data table or collection within the dataset.
- Each **Field** maps to a column within a record set and has a unique `@id`.

Let's inspect the record sets and fields available:

In [ ]:
# List all available record sets with their @id and name
record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")
for idx, rec in enumerate(record_sets):
    print(f"[{idx}] @id: {rec.id} | name: {rec.name} | description: {rec.description}")

# For demonstration, select the first record set
if record_sets:
    record_set = record_sets[0]
    print(f"\nFields in '{record_set.name}' (record_set @id: {record_set.id}):")
    for field in record_set.fields:
        print(f"    Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
else:
    raise ValueError("No record sets found in this dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For demonstration, we use the first record set. If more record sets are present, you can repeat this process for each.

In [ ]:
# Prepare dictionary of DataFrames for each record set
dataframes = {}

# List of record set ids to process (all by default)
record_set_ids = [rec.id for rec in record_sets]

for rec_id in record_set_ids:
    print(f"Loading records from record set: {rec_id}")
    df = pd.DataFrame(dataset.records(record_set=rec_id))
    dataframes[rec_id] = df
    print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")

# Example: Show first records for the first record set
first_record_set_id = record_set_ids[0]
display_cols = dataframes[first_record_set_id].columns.tolist()
print(f"Columns in record set '{first_record_set_id}': {display_cols}")

dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by categorical attributes. 

We'll select a numeric field for demonstration based on the available columns of the first record set. Change the variables as needed to explore other fields/record sets.

In [ ]:
# Choose which record set to analyze
record_set_id = first_record_set_id
df = dataframes[record_set_id]

# List all available columns
print(f"Columns in DataFrame for record set '{record_set_id}':\n{df.columns.tolist()}")

# Try to select a numeric field (common names for this dataset)
# Feel free to change these field @ids/column names as per field enumeration
numeric_candidates = [
    'percentage_coverage',            # e.g. percentage of coverage, if available
    'peptide_count',                  # e.g. peptide count field
    'MW',                             # e.g. molecular weight
    'calculated_pI',                  # e.g. isoelectric point
    'normalized_abundance'            # e.g. normalized abundance field
]
numeric_field = None
for col in df.columns:
    if col in numeric_candidates:
        numeric_field = col
        break
# If not found, try to detect the first column with subtype number/integer/float
if numeric_field is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
if numeric_field is None:
    raise ValueError("No numeric field found for EDA.")

print(f"Selected numeric field for EDA: {numeric_field}")

# Drop rows with missing values in the numeric field
filtered_df = df.dropna(subset=[numeric_field])

# Filter records where value is greater than a threshold (e.g. mean or a set minimum)
threshold = filtered_df[numeric_field].mean()
filtered = filtered_df[filtered_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}), count={len(filtered)}:")
display_cols = [numeric_field]
print(filtered[display_cols].head())

# Normalize the numeric field (z-score)
filtered[f"{numeric_field}_normalized"] = (
    filtered[numeric_field] - filtered[numeric_field].mean()
    ) / filtered[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered[[numeric_field, f"{numeric_field}_normalized"]].head())

# If a plausible group/categorical field exists, group and average
group_field_candidates = [
    'accession',
    'description',
    'sample',
    'sample_id'
]
group_field = None
for col in df.columns:
    if col in group_field_candidates:
        group_field = col
        break

if group_field:
    print(f"\nGrouping by: {group_field}")
    grouped_df = filtered.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())
else:
    print("No appropriate group field found; skipping groupby demo.")

## 5. Visualization
Visualize the distribution of the numeric field and, if grouping was possible, show mean values by group.


In [ ]:
# Visualize distribution of the selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered[numeric_field], kde=True)
plt.title(f"Distribution of '{numeric_field}' (filtered)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group_field was used, show barplot of group means
if group_field:
    plt.figure(figsize=(12, 5))
    top_groups = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    sns.barplot(x=group_field, y=numeric_field, data=top_groups)
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We loaded the dataset via its Croissant schema using `mlcroissant`, listed record sets and their fields by unique `@id` for transparency,
- Extracted records to pandas DataFrames,
- Performed exploratory analysis on a representative numeric field, including filtering, normalization, grouping, and visualization.

This approach can be adapted to explore other record sets or fields in greater detail. For further analysis, consider investigating relationships between abundance, peptide count, coverage, and sequence information as provided by this mass spectrometry dataset.